In [4]:
import os
import pandas as pd
import re

while (os.getcwd().split('\\')[len(os.getcwd().split('\\')) - 1]).upper() != "202510-MACROLP-PROYECTO":
    os.chdir("..")

print(os.getcwd())
raw = "stores\\raw\\"
work = "stores\\work\\"

c:\Users\nloza\OneDrive - Universidad de los andes\UNIVERSIDAD\9. NOVENO SEMESTRE\MACRO AVANZADA LP\202510-MacroLP-Proyecto


# Cargue de datos

In [5]:
ONET_BASE_NAMES = {
    "Abilities",
    "Skills",
    "Work Activities",
    "Work Context"
}

path_onet_abilities = f"{raw}onet\\Abilities.xlsx"
path_onet_skills = f"{raw}onet\\Abilities.xlsx"
path_onet_work_act = f"{raw}onet\\Abilities.xlsx"
path_onet_work_cont = f"{raw}onet\\Abilities.xlsx"
path_owes = f"{raw}owes\\national_m2024_dl.xlsx"

df_abilities = pd.read_excel(path_onet_abilities)
df_skills = pd.read_excel(path_onet_skills)
df_work_act = pd.read_excel(path_onet_work_act)
df_work_cont = pd.read_excel(path_onet_work_cont)
df_owes = pd.read_excel(path_owes)

# Limpiza OWES

In [6]:
df_owes.columns = (df_owes.columns.str.strip().str.lower()
                    .str.replace(" ","_",regex=False)
                    .str.replace("%","pct",regex=False))

num_cols = [c for c in df_owes.columns
            if re.match(r"^(h_|a_|jobs_|tot_emp|emp_prse|mean_prse)", c)]
for c in num_cols:
    df_owes[c] = (df_owes[c].astype(str)
                   .str.replace(r"[*,#~]","",regex=True)
                   .str.replace(",", "", regex=False)
                   .str.strip())
    df_owes[c] = pd.to_numeric(df_owes[c], errors="coerce")

df_oews = (df_owes.query("area_type==1 and o_group=='detailed'")
           .rename(columns={"occ_code":"soc"}))

df_oews["wage"] = df_oews["a_median"].fillna(df_oews["a_mean"])
df_oews = df_oews[df_oews["wage"].between(5_000, 1_000_000)]
df_oews = df_oews[["soc","occ_title","tot_emp","wage"]].dropna()

df_oews.head()

,soc,occ_title,tot_emp,wage
4,11-1011,Chief Executives,211850,206420.0
6,11-1021,General and Operations Managers,3584420,102950.0
8,11-1031,Legislators,26510,44810.0
11,11-2011,Advertising and Promotions Managers,21100,126960.0
13,11-2021,Marketing Managers,384980,161030.0


# Limpieza O*NET

Definimos diccionario `SCALES` con las escalas de relavancia según la base utilizada.
<br>
Definimos tupla `DROP_FLAGS` con las columnas que indican si la observación es relvante o debe ser suprimida.

In [ ]:
SCALES = {"abilities": "LV",
          "skills": "LV",
          "work_activities": "LV",
          "work_context": "IM"} 

DROP_FLAGS = ("recommend_suppress", "not_relevant")

Definimos funciones utilitarias para la limpieza.

In [ ]:
def norm(col):
    """snake_case compacto para cualquier encabezado."""
    return re.sub(r"[^\w]+", "_", col.strip().lower())

def col_like(df, pref):
    """primera columna cuyo nombre empieza por pref (o None)."""
    return next((c for c in df.columns if c.startswith(pref)), None)

def std_soc(arr):
    """
    Normaliza cualquier iterable (Index, Series, list) de SOC a formato 11-1011
    (quita sufijos .00 / .03, añade guion si falta, recorta extras).
    """
    s = pd.Series(arr, dtype="string")
    s = (s.str.upper()
           .str.replace(r"\.0+$", "", regex=True)
           .str.replace(r"[^\d\-]", "", regex=True))

    mask6 = s.str.match(r"^\d{6}$")
    s.loc[mask6] = s[mask6].str.replace(r"^(\d{2})(\d{4})$", r"\1-\2", regex=True)

    s = s.str.replace(r"^(\d{2}-\d{4}).*$", r"\1", regex=True)
    return s